# S1 · 01 · Tu primer agente: salida tipada
Primero el ejemplo mínimo de la **Demo 1** (un agente que responde texto y otro que extrae un `Evento`); luego tu **Ejercicio 1** (≈ 10 min).
Corre sin red ni LLM con el modelo simulado (`COURSE_OFFLINE=1`, por defecto). Para IA real: `COURSE_OFFLINE=0` en `.env` y *Restart* del kernel.


## Preparación
`preparar()` lee tu `.env` y dice qué modelo usarás. `crear_modelo` devuelve el objeto de modelo de Pydantic AI de tu ruta: **Ollama** (`gemma4:e2b`), **Gemini** o el **simulado** (no es IA: sirve para ver el recorrido sin red ni cuota).


In [ ]:
from utilidades.config import crear_modelo, describir_ruta
from utilidades.practica import preparar, revisar, ver_mensajes

settings, OFFLINE = preparar()
modelo = crear_modelo(settings, OFFLINE)
print("Modelo activo:", describir_ruta(settings, OFFLINE))


## Ejemplo mínimo 1 · Un agente que responde texto (Demo 1)
Un `Agent` son dos piezas: el **modelo** y unas **instrucciones** (cómo debe comportarse). `await agente.run(pedido)` envía el pedido al modelo y devuelve un resultado: `r.output` es la respuesta (aquí, un `str`) y `r.usage` cuenta peticiones y tokens.


In [ ]:
from pydantic_ai import Agent

agente_texto = Agent(modelo, instructions="Eres un asistente. Responde en español, en una frase.")
r = await agente_texto.run("¿Para qué sirve una agenda?")
print(r.output)
print(r.usage)


## Ejemplo mínimo 2 · Extraer datos con `output_type` (Demo 1)
Un párrafo es difícil de usar desde un programa. Con `output_type=Evento` el agente devuelve un **objeto de Pydantic validado**: el esquema de `Evento` (campos, tipos y `description`) viaja al modelo, y Pydantic AI comprueba la respuesta antes de dártela (si no cumple, le pide al modelo que la corrija).
- `Field(description=...)` es una pista para el modelo: dice qué formato queremos.
- El modelo no sabe qué día es hoy: se lo decimos en las **instrucciones** para que convierta «el jueves» en una fecha.


In [ ]:
from pydantic import BaseModel, Field


class Evento(BaseModel):
    titulo: str = Field(description="Nombre corto del evento")
    fecha: str = Field(description="Día en formato AAAA-MM-DD")
    inicio: str = Field(description="Hora de inicio HH:MM (24 horas)")
    fin: str = Field(description="Hora de fin HH:MM (24 horas)")


extractor = Agent(
    modelo,
    instructions="Extrae el evento del mensaje. Hoy es martes 2030-05-14.",
    output_type=Evento,  # la salida ya no es texto: es un Evento
)
r = await extractor.run("Reunión con Ana el jueves de 15:00 a 16:00 para revisar el presupuesto.")
print(type(r.output).__name__, r.output)
print(r.output.fecha)  # un campo, como cualquier objeto de Python
ver_mensajes(r)


`ver_mensajes` muestra el recorrido: tu pedido y la respuesta del modelo. Con el simulado los campos salen con texto de relleno (no entiende el mensaje); con IA real deberías ver `2030-05-16`, `15:00` y `16:00`. **Un esquema válido no garantiza un dato verdadero:** la validación revisa la forma; la verdad hay que comprobarla.


## Ejercicio 1 · Extrae una tarea
**Se apoya en:** Demo 1 · **Nivel:** básico · ≈ 10 min. **Qué cambia:** en la demo se extraía un `Evento`; aquí diseñas tú el contrato de una **tarea**, con un campo que solo admite tres valores.

1. Completa `Tarea` con tres campos: `titulo` (`str`), `fecha_limite` (`str` con `Field(description="AAAA-MM-DD")`) y `prioridad` (`Literal["alta", "media", "baja"]`: el modelo solo puede elegir uno de esos tres textos).
2. Ejecuta las celdas: el agente ya dice que hoy es martes 2030-05-14.

Lo esperable con IA real: `prioridad="alta"` («es urgente») y `fecha_limite="2030-05-17"` (el viernes).


In [ ]:
from typing import Literal


class Tarea(BaseModel):
    # TODO Ejercicio 1: añade titulo (str), fecha_limite (str, con Field(description="AAAA-MM-DD"))
    # y prioridad (Literal["alta", "media", "baja"]).
    pass


agente_tareas = Agent(
    modelo,
    instructions="Extrae la tarea del mensaje. Hoy es martes 2030-05-14.",
    output_type=Tarea,
)
r1 = await agente_tareas.run("Tengo que enviar el informe a Luis antes del viernes, es urgente.")
print(r1.output)


In [ ]:
revisar(
    "Ejercicio 1 · Tarea tiene titulo, fecha_limite y prioridad",
    lambda: {"titulo", "fecha_limite", "prioridad"} <= set(Tarea.model_fields),
    "Añade los tres campos a Tarea",
)
revisar(
    "Ejercicio 1 · prioridad solo admite alta, media o baja",
    lambda: Tarea.model_json_schema()["properties"]["prioridad"].get("enum") == ["alta", "media", "baja"],
    'Usa Literal["alta", "media", "baja"]',
)
revisar(
    "Ejercicio 1 · fecha_limite lleva una descripción con el formato",
    lambda: "AAAA-MM-DD" in Tarea.model_json_schema()["properties"]["fecha_limite"].get("description", ""),
    'Usa Field(description="AAAA-MM-DD")',
)
revisar(
    "Ejercicio 1 · el agente devuelve una Tarea completa",
    lambda: isinstance(r1.output, Tarea) and "prioridad" in r1.output.model_dump(),
    "Completa Tarea y vuelve a ejecutar la celda anterior",
)
if not OFFLINE and "prioridad" in Tarea.model_fields:
    # Con IA real: una pista, no una nota. Un modelo pequeño puede equivocarse con la fecha.
    print("¿prioridad alta?", r1.output.prioridad == "alta", "· ¿fecha 2030-05-17?", r1.output.fecha_limite == "2030-05-17")


## Cierre
**¿Qué puede hacer ahora Mi copiloto?** Entender un mensaje y devolver datos con forma fija (`Evento`, `Tarea`). Todavía **no consulta nada**: no ve la agenda ni el correo de Alex.
**Siguiente:** `02_herramientas.ipynb`: le damos herramientas (tools) y le pedimos permiso antes de escribir.
